# 01 — Your First PPO Ant (Exercise 1)

`scripts/train.py`, exploded cell by cell.

The script is what you use for real runs (Hydra config, multirun sweeps); this
notebook is the same thing with every intermediate step visible so you can poke
at it. By the end you will have trained a walking policy and read its curves.

**Runtime:** the training cell below is deliberately short (~2 min). A policy
that actually *walks* needs ~3M steps (~45 min on CPU) — the last section
explains how to launch that properly.

In [1]:
# Colab: uncomment to install
# !pip install "gymnasium[mujoco]" stable-baselines3 hydra-core

import sys, pathlib
# make the package importable (envs/, utils/) — notebooks live one level down
PKG = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PKG))

import numpy as np
import matplotlib.pyplot as plt
from envs import make_ant, LEG_JOINTS
print("package:", PKG)

package: /home/leander/code/rl_bootcamps/RL_bootcamp_2026/sessions/02-intermediate/crippled-ant


## 1. The environment

MuJoCo `Ant-v5`: a four-legged robot. 8 continuous actions — one torque per
joint, two joints (hip + ankle) per leg.

We drop the contact forces from the observation (`include_cfrc_ext_in_observation=False`)
and zero their reward term. This is the known-good simplification from the 2025
bootcamp: it shrinks the observation 105 → 27 and removes a notoriously noisy
reward component.

In [2]:
env = make_ant(include_cfrc_ext_in_observation=False, contact_cost_weight=0.0)

print("observation:", env.observation_space)
print("action:     ", env.action_space)
print("\nleg -> (hip, ankle) action indices:")
for leg, joints in LEG_JOINTS.items():
    print(f"  leg {leg}: {joints}")

observation: Box(-inf, inf, (27,), float64)
action:      Box(-1.0, 1.0, (8,), float32)

leg -> (hip, ankle) action indices:
  leg 0: (0, 1)
  leg 1: (2, 3)
  leg 2: (4, 5)
  leg 3: (6, 7)


### What does the reward actually contain?

Ant's reward is `forward_velocity − control_cost + healthy_bonus`. That
`healthy_bonus` (+1 per surviving step) is worth remembering: an agent that
learns to *stand still without falling* already collects ~1000 per episode.
Watch for that failure mode later — it looks like success in the reward curve.

In [3]:
obs, info = env.reset(seed=0)
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())

print(f"reward = {reward:.3f}")
for k, v in info.items():
    if "reward" in k or "cost" in k:
        print(f"  {k:>20}: {v: .3f}")

reward = -0.462
        reward_forward:  0.105
           reward_ctrl: -1.567
        reward_contact: -0.000
        reward_survive:  1.000


## 2. Training hygiene: vectorised + normalised environments

Two things every SB3 continuous-control run needs:

- **`make_vec_env(n_envs=8)`** — 8 copies stepping in parallel. PPO is
  on-policy: it throws data away after each update, so it is hungry for fresh
  samples.
- **`VecNormalize`** — Ant observations span wildly different scales (joint
  angles ~1, velocities ~10). Normalising them (and the reward) is close to
  mandatory for PPO here.

⚠️ **`VecNormalize` is part of your model.** Its running statistics must be
saved with the checkpoint and reloaded at evaluation, or the policy sees
differently-scaled inputs and silently underperforms. `scripts/train.py`
handles this; see the usage guide's warning for what it costs when you get it
wrong.

In [4]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize

env_kwargs = dict(include_cfrc_ext_in_observation=False, contact_cost_weight=0.0)

venv = make_vec_env(make_ant, n_envs=8, seed=0, env_kwargs=env_kwargs)
venv = VecNormalize(venv, norm_obs=True, norm_reward=True, gamma=0.99)

eval_venv = make_vec_env(make_ant, n_envs=1, seed=10_000, env_kwargs=env_kwargs)
eval_venv = VecNormalize(eval_venv, training=False, norm_obs=True, norm_reward=False)

print(venv)

## 3. The agent

`MlpPolicy` = a plain feed-forward net. `net_arch=[256, 256]` gives both the
actor and the critic two hidden layers of 256 units.

The other hyperparameters are the SB3-zoo defaults for Ant. Module 5 asks which
of them actually matter — don't take them on faith.

In [5]:
from stable_baselines3 import PPO

model = PPO(
    "MlpPolicy", venv,
    policy_kwargs=dict(net_arch=[256, 256]),
    learning_rate=3e-4,
    n_steps=2048,       # per env -> 8 x 2048 = 16384 steps per rollout
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=0,
    seed=123,
    tensorboard_log="../runs/notebook_tb",
)
print(model.policy)

ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=27, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=27, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=256, out_features=8, bias=True)
  (value_net): Linear(in_features=256, out_features=1, bias=True)
)


/home/leander/miniconda3/envs/rlbootcamp/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


## 4. Train (short demo run)

~2 minutes. This is **not** long enough to learn to walk — it is long enough to
see the machinery work and the reward start moving.

In [4]:
from stable_baselines3.common.evaluation import evaluate_policy

before, _ = evaluate_policy(model, eval_venv, n_eval_episodes=3, deterministic=True)
model.learn(total_timesteps=50_000, progress_bar=True)
after, _ = evaluate_policy(model, eval_venv, n_eval_episodes=3, deterministic=True)

print(f"mean return before training: {before:8.1f}")
print(f"mean return after  training: {after:8.1f}")

NameError: name 'model' is not defined

## 5. Reading the curves

TensorBoard (`tensorboard --logdir runs/`) is the real tool. The tags worth
your attention:

| tag | what it tells you |
|---|---|
| `rollout/ep_rew_mean` | the headline — is it learning at all? |
| `rollout/ep_len_mean` | episodes lengthening = falling over less |
| `train/entropy_loss` | exploration shrinking over time (expected) |
| `train/explained_variance` | is the critic any good? → 1 is ideal, ≤0 is broken |
| `train/clip_fraction` | how often PPO clips; persistently high = LR too big |

**Checkpoint questions.** Is the reward rising monotonically? What happens to
entropy over time — and what would it mean if it collapsed to zero in the first
few thousand steps?

In [3]:
# the same numbers, straight out of the SB3 logger
for tag in ["rollout/ep_rew_mean", "rollout/ep_len_mean",
            "train/entropy_loss", "train/explained_variance", "train/clip_fraction"]:
    value = model.logger.name_to_value.get(tag)
    if value is not None:
        print(f"{tag:>30}: {value: .3f}")

NameError: name 'model' is not defined

## 6. Now do it properly

Use the script for real runs — it handles run directories, evaluation
callbacks, checkpoints and the VecNormalize bookkeeping:

```bash
python scripts/train.py                       # PPO, healthy Ant, 1M steps
python scripts/train.py train.total_timesteps=3_000_000     # a real walker
python scripts/train.py algo=sac              # Exercise 2
```

**Exercise 2 — SAC vs PPO.** Train SAC for the same *wall-clock* time and
compare. SAC is off-policy: it replays old transitions, so it needs far fewer
environment steps to reach the same reward — but each step costs more compute.
Which wins per hour? Note SAC deliberately runs *without* VecNormalize (a
replay buffer full of differently-normalised transitions is a bug).

➡️ **Next:** `02_transfer_benchmark.ipynb` — take a trained policy, break its
legs, and measure what happens.